In [1]:
import os
import json
import time
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

from tqdm import tqdm 

# Viz and statistical work
from sklearn.decomposition import PCA

# Retrieval & Vector Math
import faiss

# NLP & ML
from sentence_transformers import SentenceTransformer, util, CrossEncoder
#from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments

# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Setting random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
# create a torch.device object for tensor/model placement
device = torch.device(DEVICE)

Device: cpu


In [2]:
# Extract only the text part from dataset and save it in a txt file
DATA_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
dreams_path = DATA_DIR / "dreams_emotions1.csv"
dream_text_path = DATA_DIR / "dream_text.txt"

dream_emotions_full = pd.read_csv(dreams_path)
if "dream_text" not in dream_emotions_full.columns:
    raise ValueError("dreams_emotions1.csv must contain a dream_text column")

dream_text = "\n\n".join(
    dream_emotions_full["dream_text"].dropna().astype(str).tolist()
)
dream_text_path.write_text(dream_text, encoding="utf-8")
target_word_count = int(round(dream_emotions_full["dream_text"].dropna().astype(str).str.split().str.len().mean()))
target_word_count = max(target_word_count, 1)